In [ ]:
import spacy;
import pandas as pd;
import numpy as np;
from spacy import displacy

nlp = spacy.load("en_core_web_sm")

In [ ]:

news_df = pd.read_csv("dataset/News.csv")
corpus = news_df['full_text'].tolist()

sentence = str(news_df.iloc[1]['full_text']) 
sentence_nlp = nlp(sentence)

print([(word.text, word.ent_type_) for word in sentence_nlp if word.ent_type_])

displacy.render(sentence_nlp, style="ent", jupyter=True)

In [ ]:
named_entities = []
 
for sentence in corpus:
    temp_entity_name = ''
    temp_named_entity = None
    sentence = nlp(sentence)
 
    for word in sentence:
        term = word.text
        tag = word.ent_type_
 
        if tag:
            # Build multi-word entity names
            temp_entity_name = ' '.join([temp_entity_name, term]).strip()
            temp_named_entity = (temp_entity_name, tag)
        else:
            if temp_named_entity:
                named_entities.append(temp_named_entity)
                temp_entity_name = ''
                temp_named_entity = None
 
# Build DataFrame of all named entities
entity_frame = pd.DataFrame(named_entities,
                            columns=['Entity Name', 'Entity Type'])

In [ ]:
top_entities = (entity_frame.groupby(by=['Entity Type'])
                             .size()
                             .sort_values(ascending=False)
                             .reset_index()
                             .rename(columns={0: 'Frequency'}))
 
# Show top 15 columns (transposed view)
print(top_entities.T.iloc[:, :15])

In [ ]:
from nltk.tag import StanfordNERTagger
import os
 
# Set Java path (update this to your Java installation)
java_path = r'C:\Program Files\Java\jdk1.8.0_102\bin\java.exe'
os.environ['JAVAHOME'] = java_path
 
# Initialize Stanford NER tagger (update paths to your unzipped folder)
sn = StanfordNERTagger(
     r'C:\Users\Anthony\Downloads\stanford-ner-2014-08-27 (1)\stanford-ner-2014-08-27\classifiers\english.all.3class.distsim.crf.ser.gz',
    path_to_jar=r'C:\Users\Anthony\Downloads\stanford-ner-2014-08-27 (1)\stanford-ner-2014-08-27\stanford-ner.jar'
)
 
# Tag named entities in full corpus
ner_tagged_sentences = [sn.tag(sent.split()) for sent in corpus]
 
# Extract all named entities from Stanford NER output
named_entities = []
 
for sentence in ner_tagged_sentences:
    temp_entity_name = ''
    temp_named_entity = None
 
    for term, tag in sentence:
        if tag != 'O':  # 'O' means no entity
            # Build multi-word entity names
            temp_entity_name = ' '.join([temp_entity_name, term]).strip()
            temp_named_entity = (temp_entity_name, tag)
        else:
            if temp_named_entity:
                named_entities.append(temp_named_entity)
                temp_entity_name = ''
                temp_named_entity = None

In [ ]:
named_entities = list(set(named_entities))
 
entity_frame = pd.DataFrame(named_entities,
                            columns=['Entity Name', 'Entity Type'])
 
# View top entities by name AND type
top_entities = (entity_frame.groupby(by=['Entity Name', 'Entity Type'])
                             .size()
                             .sort_values(ascending=False)
                             .reset_index()
                             .rename(columns={0: 'Frequency'}))
 
print(top_entities.head(15))
 
# View top entity types only
top_entities = (entity_frame.groupby(by=['Entity Type'])
                             .size()
                             .sort_values(ascending=False)
                             .reset_index()
                             .rename(columns={0: 'Frequency'}))
 
print(top_entities.head())